# Phase 4: Modelling (Part 2 - Baseline Support Vector Machine)
**Client:** Crédit Nationale Azur
**Objective:** Build a baseline Support Vector Machine (SVM) to predict personal loan acceptance, establishing a secondary baseline to compare against our Random Forest model.

In this notebook, we will strictly adhere to the mandatory execution pipeline: splitting the data, encoding categorical text, performing chi-squared feature selection, and finally standardising the continuous variables before training the model.

In [1]:
# Required Base Imports for Phase 4
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, chi2

## 1. Load Data and Prepare Targets
We load our cleaned dataset from the `data` directory using `joblib`. The target variable `personal_loan` is mapped to binary integers, and we drop non-predictive identifiers.

In [2]:
# Load the cleaned data using relative paths and joblib
df = joblib.load('../data/cleaned_data.pkl')

# Map the target variable exactly as specified
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})

# Separate features (X) and target (y)
# Drop both the target AND the non-predictive customer_id
X = df.drop(['personal_loan', 'customer_id'], axis=1)
y = df['personal_loan']

## 2. Train/Test Split (Pipeline Step 1)
We use an 80/20 split and apply stratification to maintain the class imbalance ratio in both sets. Explicit copies are created to prevent data leakage and Pandas warnings.

In [3]:
# Perform the 80/20 train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create explicit copies to prevent SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()

## 3. One-Hot Encoding (Pipeline Step 2)
We systematically encode all remaining categorical text variables into integer-based dummy variables. This step must precede feature selection to ensure all data is numerical.

In [4]:
# Identify ALL categorical columns containing text
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']

# Loop through and apply pd.get_dummies() strictly preventing leakage
for col in categorical_cols:
    # Transform X_train 
    dummies_train = pd.get_dummies(X_train[col], prefix=col, dtype=int)
    X_train.drop([col], axis=1, inplace=True)
    X_train = pd.concat([X_train, dummies_train], axis=1)
    
    # Transform X_test strictly after to prevent data leakage
    dummies_test = pd.get_dummies(X_test[col], prefix=col, dtype=int)
    X_test.drop([col], axis=1, inplace=True)
    X_test = pd.concat([X_test, dummies_test], axis=1)

# Align columns to guarantee train and test sets match perfectly
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print("All categorical variables successfully one-hot encoded into integers.")

All categorical variables successfully one-hot encoded into integers.


## 4. Feature Selection (Pipeline Step 3)
With a purely numerical dataset, we confidently apply `SelectKBest` with the `chi2` scoring function to extract the top 7 features before any standardisation takes place.

In [5]:
# Initialize SelectKBest with chi2 and k=7
selector = SelectKBest(score_func=chi2, k=7)

# Fit strictly on the training data, then transform both
X_train_selected_array = selector.fit_transform(X_train, y_train)
X_test_selected_array = selector.transform(X_test)

# Cross-reference the headless Numpy array back to Pandas column names
selected_mask = selector.get_support()
selected_features = X_train.columns[selected_mask]

# Reconstruct DataFrames with the selected features
X_train_selected = pd.DataFrame(X_train_selected_array, columns=selected_features, index=X_train.index)
X_test_selected = pd.DataFrame(X_test_selected_array, columns=selected_features, index=X_test.index)

print(f"Selected Features: {list(selected_features)}")

Selected Features: ['yrs_experience', 'income', 'mortgage_amt', 'fixed_deposit_acct', 'education_level_Advanced or Professional', 'education_level_Graduate', 'education_level_Undergraduate']


## 5. Standardisation (Pipeline Step 4)
We apply `StandardScaler` exclusively to the continuous features that survived the selection step, applying strict 2D reshaping.

In [6]:
# Define continuous features
continuous_features = [
    'age', 'yrs_experience', 'family_size', 'income', 
    'mortgage_amt', 'credit_card_spend'
]

# Filter for continuous columns that were retained during selection
cols_to_scale = [col for col in continuous_features if col in selected_features]

# Scale selected continuous columns
for col in cols_to_scale:
    scaler = StandardScaler()
    
    # Fit strictly on training data with 2D reshape
    scaler.fit(X_train_selected[col].values.reshape(-1, 1))
    
    # Transform both train and test sets
    X_train_selected[col] = scaler.transform(X_train_selected[col].values.reshape(-1, 1)).flatten()
    X_test_selected[col] = scaler.transform(X_test_selected[col].values.reshape(-1, 1)).flatten()
    
print("Standardisation complete.")

Standardisation complete.


## 6. Train Baseline Model and Evaluate (Pipeline Step 5)
We instantiate our Baseline Support Vector Machine, train it on the rigorously prepared data, and evaluate its performance against the class imbalance.

In [7]:
# Instantiate and fit the baseline SVM model
svm_model = SVC(random_state=42)
svm_model.fit(X_train_selected, y_train)

# Generate predictions on the test set
y_pred = svm_model.predict(X_test_selected)

# Flatten the confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# Calculate specific metrics for imbalanced classes
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Output the evaluation metrics
print("Baseline Support Vector Machine Evaluation")
print("*" * 42)
print(f"True Negatives (TN):  {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP):  {tp}")
print("*" * 42)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

Baseline Support Vector Machine Evaluation
******************************************
True Negatives (TN):  976
False Positives (FP): 44
False Negatives (FN): 62
True Positives (TP):  118
******************************************
Precision: 0.7284
Recall:    0.6556
F1-Score:  0.6901
